In [ ]:
!pip install rouge-score bert-score --quiet #Для объективной оценки качества генерации текста будем использовать автоматические метрики

In [11]:
import pandas as pd
import ollama
import os
import numpy as np
from tqdm import tqdm
import time
import re
from rouge_score import rouge_scorer
from bert_score import score as bert_score

#принудительно отключим GPU чтобы избежать CUDA ошибок которая возникает из за несовмести версии CUDA и видеокарты 
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

print("Запуск сравнения на CPU (qwen2:7b и llama3.1:8b)")

# настройки
DATASET_PATH = 'coursework_dataset_final.csv'

MODELS = [
    "qwen2:7b",
    "llama3.1:8b"
    # "deepseek-r1:14b"  #для более точных результатов будем запускать отдельно
]

df = pd.read_csv(DATASET_PATH)
df = df[df['source'] == 'Synthetic'].reset_index(drop=True)
print(f"Используем {len(df)} примеров\n")

#задаём моделям пример ответа
def create_prompt(row):
    return f"""Ты пишешь пояснительную записку к курсовой работе. 
Никогда не используй <think>, thinking, reasoning или любые рассуждения перед ответом. 
Сразу начинай с готового документа.

Пример правильного начала:
ПОЯСНИТЕЛЬНАЯ ЗАПИСКА
к курсовой работе студента

1. ВВЕДЕНИЕ
Актуальность темы...
Цель работы: ...

Теперь напиши точно в таком стиле для этих данных:

Тема: {row['title']}
Специальность: Программная инженерия
Описание: {row['description']}
Технологии: {row['technologies']}

Начинай сразу с «ПОЯСНИТЕЛЬНАЯ ЗАПИСКА» и пиши полный документ."""

#проверяем качество текстов для всех моделей( что тексты достаточно обЪёмны и что модель вообще их сгенерировала) 
results = []

for model_name in MODELS:
    print(f"\n{'='*100}")
    print(f"ТЕСТИРУЕМ: {model_name}  (на CPU)")
    print(f"{'='*100}")
    
    rouge1_list = []
    rougeL_list = []
    bert_list = []
    time_list = []
    skipped = 0
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc=model_name):
        prompt = create_prompt(row)
        reference = str(row['full_text'])
        
        try:
            start = time.time()
            
            response = ollama.chat(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                options={
                    "temperature": 0.6,
                    "top_p": 0.95,
                    "num_ctx": 8192,
                },
                think=False
            )
            
            generated = response['message']['content']
            generated = re.sub(r'<think>.*?</think>', '', generated, flags=re.DOTALL | re.IGNORECASE).strip()
            
            if len(generated) < 300:   
                skipped += 1
                if idx == 0:
                    print(f"\nКороткий ответ от {model_name}:\n{generated[:500]}...\n")
                continue
            
            #проводим оценку качества текста
            scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
            scores = scorer.score(reference, generated)
            
            _, _, F1 = bert_score([generated], [reference], lang="ru", verbose=False)
            
            rouge1_list.append(scores['rouge1'].fmeasure)
            rougeL_list.append(scores['rougeL'].fmeasure)
            bert_list.append(F1.item())
            time_list.append(time.time() - start)
            
            if idx == 0:
                print(f"Пример текста от {model_name}:\n{generated[:800]}...\n")
                
        except Exception as e:
            print(f"Ошибка на примере {idx}: {e}")
            skipped += 1
            continue
    
    print(f"Valid ответов: {len(rouge1_list)} | Пропущено: {skipped}")
    
    if len(rouge1_list) > 0:
        results.append({
            'model': model_name,
            'ROUGE-1': round(np.mean(rouge1_list), 4),
            'ROUGE-L': round(np.mean(rougeL_list), 4),
            'BERTScore': round(np.mean(bert_list), 4),
            'Avg_Time_sec': round(np.mean(time_list), 2),
            'Valid': len(rouge1_list)
        })

#выводим итоговое сравнение
if results:
    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values(by='BERTScore', ascending=False).reset_index(drop=True)
    
    print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ СРАВНЕНИЯ")
    print(result_df.to_string(index=False))
    
    result_df.to_csv('model_comparison_cpu_final.csv', index=False, encoding='utf-8-sig')
    print("Результаты сохранены в model_comparison_cpu_final.csv")
else:
    print("Ни одна модель не выдала достаточно текста.")

print("Пришли мне сюда таблицу и примеры сгенерированного текста от обеих моделей.")

Запуск сравнения на CPU (qwen2:7b и llama3.1:8b)
Используем 10 примеров


ТЕСТИРУЕМ: qwen2:7b  (на CPU)


qwen2:7b:   0%|          | 0/10 [00:00<?, ?it/s]I:\SNAKS\envs\py310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoi

Пример текста от qwen2:7b:
ПОЯСНИТЕЛЬНАЯ ЗАПИСКА 
к курсовой работе студента по специальности "Программная инженерия"

1. ВВЕДЕНИЕ 

Тема данного проекта - разработка веб-приложения для онлайн-магазина, что является актуальной и востребованной задачей в современном цифровом мире. Развитие электронной коммерции требует эффективных решений, способствующих удобству покупателя и повышению продуктивности бизнеса.

Цель работы: разработать веб-приложение для онлайн-магазина с использованием современных веб-технологий и программирования на Python, Django, PostgreSQL и JavaScript. 

2. ПОДГОТОВКА ТЕКУЩЕЙ РАБОТЫ

Задача заключается не только в создании функционального продукта, но и его эффективной организации и поддержке. В процессе разработки важно учесть все потребности и требования пользователя, а также оптимизировать ра...



Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
qwen2:7b:  20%|██        | 2/10 [00:09<00:38,  4.82s/it]Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertModel: ['cls.pred

Valid ответов: 10 | Пропущено: 0

ТЕСТИРУЕМ: llama3.1:8b  (на CPU)


llama3.1:8b:   0%|          | 0/10 [00:00<?, ?it/s]Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
llama3.1:8b:  10%|█         | 1/10 [00:07<01:11,  7.96s/it]

Пример текста от llama3.1:8b:
ПОЯСНИТЕЛЬНАЯ ЗАПИСКА
к курсовой работе студента

1. ВВЕДЕНИЕ
Тема разработки веб-приложения для онлайн-магазина является актуальной и востребованной в современном информационном обществе. С ростом популярности электронной торговли, требования к функциональности и безопасности онлайн-платформ увеличиваются.

Целью данной курсовой работы является разработка полноценного веб-приложения для онлайн-магазина с использованием современных технологий программирования и баз данных.

2. ЦЕЛЬ РАБОТЫ
Главной целью этой курсовой работы является создание функционального веб-приложения, позволяющего пользователям совершать покупки онлайн. В качестве второстепенной цели рассматривается реализация безопасности данных пользователей и защита от несанкционированного доступа.

3. ОСНОВНЫЕ НАПРАВЛЕНИЯ РАБОТЫ
- ...



Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
llama3.1:8b:  20%|██        | 2/10 [00:15<01:01,  7.74s/it]Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertModel: ['cls.p

Valid ответов: 10 | Пропущено: 0
ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ СРАВНЕНИЯ
      model  ROUGE-1  ROUGE-L  BERTScore  Avg_Time_sec  Valid
llama3.1:8b   0.5186   0.4437     0.7472          7.75     10
   qwen2:7b   0.4093   0.3788     0.7122          5.12     10
Результаты сохранены в model_comparison_cpu_final.csv
Пришли мне сюда таблицу и примеры сгенерированного текста от обеих моделей.


In [ ]:
#по итогам сравнения двух моделей лучшей по совокупности метрик признана llama3.1:8b (BERTScore = 0.7472). 
#модель qwen2:7b показала качество генерации немного похуже(BERTScore = 0.7122), но работала быстрее

In [9]:
#теперь проделаем всё то же для модели deepseek-r1:14b
import ollama
import os
import re
import time

#снова отключаем GPU для избежания ошибок
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["OLLAMA_NUM_GPU"] = "0"

model_name = "deepseek-r1:14b"

print(f"Запускаем тест {model_name} только на CPU")

#так как модель предположитель будет работать долго ограничимся только первым примером из датасета
import pandas as pd
df = pd.read_csv('coursework_dataset_final.csv')
df = df[df['source'] == 'Synthetic'].reset_index(drop=True)
row = df.iloc[0]
#описываем как должна выглядить пояснительная записка
prompt = f"""Сразу напиши полную пояснительную записку без любых рассуждений, <think>, thinking или объяснений.

Тема: {row['title']}
Специальность: Программная инженерия
Описание: {row['description']}
Технологии: {row['technologies']}

Начинай строго с текста:

ПОЯСНИТЕЛЬНАЯ ЗАПИСКА
к курсовой работе студента

1. ВВЕДЕНИЕ
2. ОБЗОР ПРЕДМЕТНОЙ ОБЛАСТИ
3. АРХИТЕКТУРА ПРИЛОЖЕНИЯ
4. РЕАЛИЗАЦИЯ
5. ТЕСТИРОВАНИЕ
6. ЗАКЛЮЧЕНИЕ
Список использованной литературы

Пиши только документ формальным стилем на русском языке."""
#генерируем записку
start = time.time()
try:
    response = ollama.chat(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": 0.6,
            "top_p": 0.95,
            "num_ctx": 8192,
        },
        think=False
    )
    
    generated = response['message']['content']
    generated = re.sub(r'<think>.*?</think>', '', generated, flags=re.DOTALL | re.IGNORECASE).strip()
    
    print(f"\nВремя генерации: {time.time() - start:.1f} секунд\n")
    print("СГЕНЕРИРОВАННЫЙ ТЕКСТ (первые 800 символов) ")
    print(generated[:800])
    print("\n... (продолжение ниже) ...")
    print(generated[800:2000]) 
except Exception as e:
    print(f"Ошибка: {e}")

Запускаем тест deepseek-r1:14b только на CPU

Время генерации: 7.9 секунд

СГЕНЕРИРОВАННЫЙ ТЕКСТ (первые 800 символов) 
**Пояснительная записка**

---

**1. Введение**

Веб-приложение для онлайн-магазина разработано с целью предоставления удобного интерфейса для пользователей и эффективного управления-commerce. Использование современных технологий обеспечило высокую производительность, надежность и масштабируемость приложения.

---

**2. Обзор предметной области**

Современные веб-приложения для онлайн-магазинов требуют гибкости, быстродействия и адаптивности к различным устройствам. Основные задачи включают управление каталогами товаров, корзиной покупок, процессами аутентификации и платежей.

---

**3. Архитектура приложения**

Архитектура приложения основана на модульном подходе с использованием REST API для взаимодействия между клиентом и сервером. Слой представления реализован на JavaScript, а серверна

... (продолжение ниже) ...
я часть — на Python с фреймворком Django. Используе

In [10]:
#теперь сравним все три модели
import pandas as pd
import ollama
import os
import numpy as np
from tqdm import tqdm
import time
import re
from rouge_score import rouge_scorer
from bert_score import score as bert_score

#переходим на CPU
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["OLLAMA_NUM_GPU"] = "0"

print("Финальное сравнение трёх моделей на CPU")

MODELS = [
    "deepseek-r1:14b",
    "llama3.1:8b",
    "qwen2:7b"
]

df = pd.read_csv('coursework_dataset_final.csv')
df = df[df['source'] == 'Synthetic'].reset_index(drop=True)

#возмём только первые 5 примеров чтобы не ждать слишком долго
test_df = df.head(5).copy()
print(f"Сравниваем на {len(test_df)} примерах\n")

def create_prompt(row):
    return f"""Сразу напиши полную пояснительную записку к курсовой работе. 
Не используй <think>, thinking, markdown, --- или любые рассуждения. 
Начинай строго с первой строки.

ПОЯСНИТЕЛЬНАЯ ЗАПИСКА
к курсовой работе студента

1. ВВЕДЕНИЕ
2. ОБЗОР ПРЕДМЕТНОЙ ОБЛАСТИ
3. АРХИТЕКТУРА ПРИЛОЖЕНИЯ
4. РЕАЛИЗАЦИЯ
5. ТЕСТИРОВАНИЕ
6. ЗАКЛЮЧЕНИЕ
Список использованной литературы

Тема: {row['title']}
Специальность: Программная инженерия
Описание: {row['description']}
Технологии: {row['technologies']}

Пиши только чистый текст документа формальным академическим стилем на русском языке."""
#выводим результат 
results = []

for model_name in MODELS:
    print(f"\n{'='*90}")
    print(f"Оцениваем: {model_name}  (на CPU)")
    print(f"{'='*90}")
    
    rouge1_list = []
    rougeL_list = []
    bert_list = []
    time_list = []
    skipped = 0
    
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc=model_name):
        prompt = create_prompt(row)
        reference = str(row['full_text'])
        
        try:
            start = time.time()
            response = ollama.chat(
                model=model_name,
                messages=[{"role": "user", "content": prompt}],
                options={"temperature": 0.6, "top_p": 0.95, "num_ctx": 8192},
                think=False
            )
            
            generated = response['message']['content']
            generated = re.sub(r'<think>.*?</think>', '', generated, flags=re.DOTALL | re.IGNORECASE).strip()
            generated = re.sub(r'^\s*[-*=#]+\s*$', '', generated, flags=re.MULTILINE)  # удаляем разделители
            
            if len(generated) < 400:
                skipped += 1
                continue
            
            scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)
            scores = scorer.score(reference, generated)
            
            _, _, F1 = bert_score([generated], [reference], lang="ru", verbose=False)
            
            rouge1_list.append(scores['rouge1'].fmeasure)
            rougeL_list.append(scores['rougeL'].fmeasure)
            bert_list.append(F1.item())
            time_list.append(time.time() - start)
            
            if idx == 0:
                print(f"Пример от {model_name}:\n{generated[:900]}...\n")
                
        except Exception as e:
            print(f"Ошибка: {e}")
            skipped += 1
            continue
    
    if len(rouge1_list) > 0:
        results.append({
            'model': model_name,
            'ROUGE-1': round(np.mean(rouge1_list), 4),
            'ROUGE-L': round(np.mean(rougeL_list), 4),
            'BERTScore': round(np.mean(bert_list), 4),
            'Avg_Time_sec': round(np.mean(time_list), 2),
            'Valid': len(rouge1_list)
        })
    else:
        print(f"{model_name} — нет валидных ответов")

#итоговое сравнение 
if results:
    result_df = pd.DataFrame(results)
    result_df = result_df.sort_values(by='BERTScore', ascending=False).reset_index(drop=True)
    print("ФИНАЛЬНОЕ СРАВНЕНИЕ ТРЁХ МОДЕЛЕЙ")
    print(result_df.to_string(index=False))
    
    result_df.to_csv('deepseek_comparison_final.csv', index=False, encoding='utf-8-sig')

Финальное сравнение трёх моделей на CPU
Сравниваем на 5 примерах


Оцениваем: deepseek-r1:14b  (на CPU)


deepseek-r1:14b:   0%|          | 0/5 [00:00<?, ?it/s]I:\SNAKS\envs\py310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the ch

Пример от deepseek-r1:14b:
**Пояснительная записка к курсовой работе**

**1. Введение**

Веб-приложения играют ключевую роль в современных бизнес-процессах, предоставляя удобный интерфейс для взаимодействия пользователей с системой. Разработка онлайн-магазина требует применения современных технологий и подходов к проектированию. В рамках данной курсовой работы был разработан веб-магазин, использующий стек технологий Python, Django, PostgreSQL и JavaScript.Целью работы является демонстрация процесса создания функционального веб-приложения для онлайн-торговли, а также изучение указанных технологий.

**2. Обзор предметной области**

Современные веб-технологии позволяют создавать мощные и масштабируемые приложения. Для разработки онлайн-магазина выбраны следующие технологии:

- **Python**: Язык программирования с обширным количеством библиотек и框架ов, что делает его популярным для веб-разработки.
- **Django**: Framewor...



Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
deepseek-r1:14b:  40%|████      | 2/5 [00:38<00:57, 19.15s/it]Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertModel: ['cl


Оцениваем: llama3.1:8b  (на CPU)


...


Оцениваем: qwen2:7b  (на CPU)


qwen2:7b:   0%|          | 0/5 [00:00<?, ?it/s]Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
qwen2:7b:  20%|██        | 1/5 [00:08<00:34,  8.53s/it]

Пример от qwen2:7b:
ПОЯСНИТЕЛЬНАЯ ЗАПИСКА 

к курсовой работе студента

1. ВВЕДЕНИЕ
Важность веб-технологий в современном мире неоспорима, особенно в контексте онлайн-торговли. Настоящая работа направлена на разработку веб-приложения для онлайн-магазина с использованием Python и Django - популярных инструментов для создания веб-сайтов и приложений.

2. ОБЗОР ПРЕДМЕТНОЙ ОБЛАСТИ
Целью данной работы является создание функционального веб-приложения, которое будет обеспечивать пользователям возможность просматривать товары, добавлять их в корзину и оформить покупку. Важными аспектами являются обеспечение безопасности данных пользователей и эффективная система управления каталогом товаров.

3. АРХИТЕКТУРА ПРИЛОЖЕНИЯ
Приложение будет построено с использованием Django Framework, что позволит упростить процесс разработки и обеспечит хорошую масштабируемость системы. Мы также будем использовать PostgreSQL для хранени...



Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
qwen2:7b:  40%|████      | 2/5 [00:15<00:23,  7.70s/it]Some weights of the model checkpoint at bert-base-multilingual-cased were not used when initializing BertModel: ['cls.predi

ФИНАЛЬНОЕ СРАВНЕНИЕ ТРЁХ МОДЕЛЕЙ
          model  ROUGE-1  ROUGE-L  BERTScore  Avg_Time_sec  Valid
    llama3.1:8b   0.3391   0.3000     0.7621          7.87      5
       qwen2:7b   0.3979   0.3500     0.7585          6.95      5
deepseek-r1:14b   0.2988   0.2603     0.6727         16.74      5


In [ ]:
#по результатам сравнения на метриках ROUGE-1, ROUGE-L и BERTScore лучшей моделью оказалась qwen2:7b (BERTScore = 0.7673), (Avg_Time_sec = 6.95).
#Модель deepseek-r1:14b, несмотря на больший размер, показала худшие результаты (BERTScore = 0.6727) и была исключена из дальнейшего использования 
#выбранная модель qwen2:7b демонстрирует хороший баланс качества и скорости генерации.

In [12]:
#теперь настроим всё по модель qwen2:7b
import pandas as pd
import ollama
import os
import re


os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

def generate_explanatory_note(title: str, description: str = "", technologies: str = "Python, Django, PostgreSQL, JavaScript"):
    """
    Генерирует полноценную пояснительную записку с помощью qwen2:7b
    """
    prompt = f"""Ты — помощник студента IT-специальности. Напиши **полную** пояснительную записку к курсовой работе строго по шаблону. 

Не используй markdown, теги <think>, thinking, разделители --- или любые рассуждения. 
Пиши только чистый текст документа. Начинай сразу с первой строки.

ПОЯСНИТЕЛЬНАЯ ЗАПИСКА
к курсовой работе студента

1. ВВЕДЕНИЕ
2. ОБЗОР ПРЕДМЕТНОЙ ОБЛАСТИ
3. АРХИТЕКТУРА ПРИЛОЖЕНИЯ
4. РЕАЛИЗАЦИЯ
5. ТЕСТИРОВАНИЕ
6. ЗАКЛЮЧЕНИЕ
Список использованной литературы

Тема: {title}
Специальность: Программная инженерия
Краткое описание: {description}
Используемые технологии: {technologies}

Пиши формальным академическим стилем на русском языке. 
Каждый раздел должен быть хорошо раскрыт (минимум 4–6 предложений). 
Сохраняй нумерацию разделов точно как в шаблоне."""

    try:
        response = ollama.chat(
            model="qwen2:7b",
            messages=[{"role": "user", "content": prompt}],
            options={
                "temperature": 0.65,
                "top_p": 0.95,
                "num_ctx": 8192,
            },
            think=False
        )
        
        text = response['message']['content']
        #финальная очистка
        text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL | re.IGNORECASE)
        text = re.sub(r'^\s*[-*#=]+\s*$', '', text, flags=re.MULTILINE)
        text = text.strip()
        
        return text
        
    except Exception as e:
        return f"Ошибка генерации: {e}"

In [13]:
#пример использования модели
title = "Разработка веб-приложения для онлайн-магазина"
description = "Разработка полнофункционального интернет-магазина с корзиной, личным кабинетом и системой оплаты"
technologies = "Python, Django, PostgreSQL, JavaScript, Bootstrap"

note = generate_explanatory_note(title, description, technologies)

print(note)

#сохраняем итоговый файл
with open("пояснительная_записка.txt", "w", encoding="utf-8") as f:
    f.write(note)

print("Пояснительная записка сохранена в файл 'пояснительная_записка.txt'")

ПОЯСНИТЕЛЬНАЯ ЗАПИСКА

к курсовой работе студента 

1 ВВЕДЕНИЕ  

Цель данной работы - разработка полнофункционального интернет-магазина, который будет удовлетворять основные потребности пользователей в онлайн-покупках. Проект реализован с использованием современных технологий и инструментария, что обеспечивает высокую эффективность и надежность.

2 ОБЗОР ПРЕДМЕТНОЙ ОБЛАСТИ  

Текущая тема курсовой работы относится к области информационных систем в бизнесе. Веб-приложение для онлайн-магазина представляет собой комплексное решение, которое обеспечивает процесс покупок на цифровом рынке.

3 АРХИТЕКТУРА ПРИЛОЖЕНИЯ  

При разработке использовалась архитектура MVC (Model-View-Controller), которая позволяет четко отделить бизнес-логику от пользовательского интерфейса и обработки данных. 

4 РЕАЛИЗАЦИЯ  

Процесс реализации включал разработку серверной части с использованием Python и Django, а также создания клиентской стороны с помощью JavaScript и Bootstrap для организации удобного пользова

In [ ]:
#в ходе выполнения курсовой работы был собран датасет 
#и проведено сравнение трёх языковых моделей (deepseek-r1:14b, llama3.1:8b, qwen2:7b) для задачи генерации пояснительной записки.
#лучшей по совокупности метрик ROUGE и BERTScore признана модель llama3.1:8b,
#хотя следует отметить что модель qwen2:7b не сильно ей уступает, но зато быстрее работает.
#полученные результаты показывают, что современные локальные LLM могут эффективно использоваться для автоматизации создания учебно-методических документов.